# Full Matplotlib EDA (All Available CSVs)

This notebook auto-discovers all CSV files from `data/raw` or `data/processed` and generates comprehensive table-level EDA plots using **Matplotlib**.

- Numeric: histogram + boxplot
- Categorical: top-value bar chart
- Datetime: daily record counts + metric trends
- Table-level numeric correlation heatmap

All figures are saved under `images/eda/full_matplotlib/<table_name>/`.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 140
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

MAX_CATEGORICAL_UNIQUE = 60
TOP_K_CATEGORIES = 20
MAX_TIME_METRICS_PER_TABLE = 10

DATE_NAME_HINTS = ('date', 'time', 'month', 'year', 'day')
IMAGE_ROOT = Path('..') / 'images' / 'eda' / 'full_matplotlib'
IMAGE_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Image output root: {IMAGE_ROOT.resolve()}')

Image output root: C:\Users\ADMIN\Documents\learning - projects\DATATHON\datathon-hkbaleycb4\images\eda\full_matplotlib


In [2]:
def detect_data_base_dir() -> Path:
    candidates = [
        Path('..') / 'data' / 'raw',
        Path('data') / 'raw',
        Path('..') / 'data' / 'processed',
        Path('data') / 'processed',
    ]
    for c in candidates:
        if c.exists() and c.is_dir() and any(c.glob('*.csv')):
            return c.resolve()
    for c in candidates:
        if c.exists() and c.is_dir():
            return c.resolve()
    raise FileNotFoundError('No data directory found. Expected data/raw or data/processed.')


def list_csv_files(data_dir: Path) -> list[Path]:
    files = sorted(data_dir.glob('*.csv'))
    if not files:
        print(f'No CSV files found in: {data_dir}')
    return files


def maybe_parse_datetimes(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        lower = str(col).lower()
        if any(h in lower for h in DATE_NAME_HINTS):
            converted = pd.to_datetime(out[col], errors='coerce', utc=False)
            if converted.notna().sum() > 0:
                out[col] = converted
    return out


def load_all_tables(data_dir: Path) -> dict[str, pd.DataFrame]:
    tables = {}
    for path in list_csv_files(data_dir):
        try:
            df = pd.read_csv(path, low_memory=False)
            df = maybe_parse_datetimes(df)
            tables[path.stem] = df
            print(f'Loaded {path.name}: shape={df.shape}')
        except Exception as exc:
            print(f'Failed to load {path.name}: {exc}')
    return tables


DATA_DIR = detect_data_base_dir()
TABLES = load_all_tables(DATA_DIR)
print(f'Using DATA_DIR: {DATA_DIR}')
print(f'Total loaded tables: {len(TABLES)}')

Loaded customers.csv: shape=(121930, 7)
Loaded geography.csv: shape=(39948, 4)
Loaded inventory.csv: shape=(60247, 17)
Loaded order_items.csv: shape=(714669, 7)
Loaded orders.csv: shape=(646945, 8)
Loaded payments.csv: shape=(646945, 4)
Loaded products.csv: shape=(2412, 8)
Loaded promotions.csv: shape=(50, 10)
Loaded returns.csv: shape=(39939, 7)
Loaded reviews.csv: shape=(113551, 7)
Loaded sales.csv: shape=(3833, 3)
Loaded sample_submission.csv: shape=(548, 3)
Loaded shipments.csv: shape=(566067, 4)
Loaded web_traffic.csv: shape=(3652, 7)
Using DATA_DIR: C:\Users\ADMIN\Documents\learning - projects\DATATHON\datathon-hkbaleycb4\data\raw
Total loaded tables: 14


In [3]:
def select_column_groups(df: pd.DataFrame):
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    datetime_cols = df.select_dtypes(include=['datetime64[ns]', 'datetime64[us]', 'datetimetz']).columns.tolist()
    categorical_cols = [
        c for c in df.columns
        if c not in numeric_cols and c not in datetime_cols
    ]
    return numeric_cols, categorical_cols, datetime_cols


def save_figure(fig, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path, bbox_inches='tight')
    plt.close(fig)


def plot_numeric_distributions(df: pd.DataFrame, table_name: str, numeric_cols: list[str], out_dir: Path):
    for col in numeric_cols:
        series = pd.to_numeric(df[col], errors='coerce').dropna()
        if series.empty:
            continue

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].hist(series, bins=40)
        axes[0].set_title(f'{table_name}.{col} - Histogram')
        axes[0].set_xlabel(col)
        axes[0].set_ylabel('Frequency')

        axes[1].boxplot(series, vert=False)
        axes[1].set_title(f'{table_name}.{col} - Boxplot')
        axes[1].set_xlabel(col)

        save_figure(fig, out_dir / f'numeric_{col}.png')


def plot_categorical_counts(df: pd.DataFrame, table_name: str, categorical_cols: list[str], out_dir: Path):
    for col in categorical_cols:
        vc = df[col].astype('string').fillna('NULL').value_counts(dropna=False)
        if vc.empty:
            continue
        if vc.shape[0] > MAX_CATEGORICAL_UNIQUE:
            vc = vc.head(TOP_K_CATEGORIES)

        fig, ax = plt.subplots(figsize=(12, 5))
        ax.bar(vc.index.astype(str), vc.values)
        ax.set_title(f'{table_name}.{col} - Value Counts (top {len(vc)})')
        ax.set_xlabel(col)
        ax.set_ylabel('Count')
        ax.tick_params(axis='x', rotation=45)
        save_figure(fig, out_dir / f'categorical_{col}.png')


def plot_datetime_patterns(df: pd.DataFrame, table_name: str, datetime_cols: list[str], numeric_cols: list[str], out_dir: Path):
    for dt_col in datetime_cols:
        dt_series = pd.to_datetime(df[dt_col], errors='coerce').dropna()
        if dt_series.empty:
            continue

        daily_count = dt_series.dt.floor('D').value_counts().sort_index()
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.plot(daily_count.index, daily_count.values)
        ax.set_title(f'{table_name}.{dt_col} - Daily Record Count')
        ax.set_xlabel('Date')
        ax.set_ylabel('Count')
        save_figure(fig, out_dir / f'time_count_{dt_col}.png')

        usable_metrics = [c for c in numeric_cols if c != dt_col][:MAX_TIME_METRICS_PER_TABLE]
        if not usable_metrics:
            continue

        tmp = df[[dt_col] + usable_metrics].copy()
        tmp[dt_col] = pd.to_datetime(tmp[dt_col], errors='coerce').dt.floor('D')
        tmp = tmp.dropna(subset=[dt_col])
        grouped = tmp.groupby(dt_col, as_index=True).mean(numeric_only=True).sort_index()
        if grouped.empty:
            continue

        for metric in grouped.columns:
            fig, ax = plt.subplots(figsize=(12, 4))
            ax.plot(grouped.index, grouped[metric].values)
            ax.set_title(f'{table_name} - Daily Mean {metric} by {dt_col}')
            ax.set_xlabel(dt_col)
            ax.set_ylabel(metric)
            save_figure(fig, out_dir / f'time_mean_{dt_col}_{metric}.png')


def plot_correlation_heatmap(df: pd.DataFrame, table_name: str, numeric_cols: list[str], out_dir: Path):
    if len(numeric_cols) < 2:
        return
    corr = df[numeric_cols].corr(numeric_only=True)
    if corr.empty:
        return

    fig, ax = plt.subplots(figsize=(max(8, len(numeric_cols) * 0.6), max(6, len(numeric_cols) * 0.6)))
    im = ax.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr.columns)))
    ax.set_yticks(range(len(corr.index)))
    ax.set_xticklabels(corr.columns, rotation=90)
    ax.set_yticklabels(corr.index)
    ax.set_title(f'{table_name} - Numeric Correlation Heatmap')
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label('Correlation')
    save_figure(fig, out_dir / 'numeric_correlation_heatmap.png')

In [4]:
summary_rows = []

for table_name, df in TABLES.items():
    out_dir = IMAGE_ROOT / table_name
    numeric_cols, categorical_cols, datetime_cols = select_column_groups(df)

    summary_rows.append(
        {
            'table': table_name,
            'rows': int(df.shape[0]),
            'cols': int(df.shape[1]),
            'numeric_cols': len(numeric_cols),
            'categorical_cols': len(categorical_cols),
            'datetime_cols': len(datetime_cols),
            'missing_cells': int(df.isna().sum().sum()),
        }
    )

    print(f'\n=== {table_name} ===')
    print(f'shape={df.shape}, numeric={len(numeric_cols)}, categorical={len(categorical_cols)}, datetime={len(datetime_cols)}')

    plot_numeric_distributions(df, table_name, numeric_cols, out_dir)
    plot_categorical_counts(df, table_name, categorical_cols, out_dir)
    plot_datetime_patterns(df, table_name, datetime_cols, numeric_cols, out_dir)
    plot_correlation_heatmap(df, table_name, numeric_cols, out_dir)

summary_df = pd.DataFrame(summary_rows).sort_values('table').reset_index(drop=True)
display(summary_df)

summary_path = IMAGE_ROOT / 'table_summary.csv'
summary_df.to_csv(summary_path, index=False)
print(f'\nSaved table summary: {summary_path.resolve()}')

ValueError: 'datetime64[us]' is too specific of a frequency, try passing 'datetime64'

In [ ]:
# Optional: explicit business-level plots for common Datathon files if present.

if 'sales' in TABLES and {'Date', 'Revenue'}.issubset(TABLES['sales'].columns):
    sales = TABLES['sales'].copy()
    sales['Date'] = pd.to_datetime(sales['Date'], errors='coerce')
    sales = sales.dropna(subset=['Date']).sort_values('Date')
    fig, ax = plt.subplots(figsize=(13, 4))
    ax.plot(sales['Date'], sales['Revenue'])
    ax.set_title('Sales Revenue Over Time')
    ax.set_xlabel('Date')
    ax.set_ylabel('Revenue')
    save_figure(fig, IMAGE_ROOT / 'sales_revenue_over_time.png')

if 'web_traffic' in TABLES and 'date' in TABLES['web_traffic'].columns:
    wt = TABLES['web_traffic'].copy()
    wt['date'] = pd.to_datetime(wt['date'], errors='coerce')
    wt = wt.dropna(subset=['date']).sort_values('date')
    numeric_wt = wt.select_dtypes(include=['number']).columns.tolist()
    for col in numeric_wt:
        fig, ax = plt.subplots(figsize=(13, 4))
        ax.plot(wt['date'], wt[col])
        ax.set_title(f'Web Traffic - {col} Over Time')
        ax.set_xlabel('Date')
        ax.set_ylabel(col)
        save_figure(fig, IMAGE_ROOT / f'web_traffic_{col}_over_time.png')

print('Finished generating full Matplotlib EDA outputs.')